# Bank transaction parser: QLoRA fine-tune on a free Kaggle T4

Runs the whole experiment end to end: regex baseline, base-model few-shot baseline, QLoRA fine-tune of Qwen2.5-1.5B-Instruct, evaluation on seen and unseen formats/merchants, and upload to the Hugging Face Hub.

**Settings:** Accelerator = GPU T4 x2 (one is used), Internet = on. For the upload step add a secret named `HF_TOKEN` (Add-ons → Secrets) holding a Hugging Face *write* token.

Code: https://github.com/Tanzil-Ali-Rizvi/bank-txn-parser-qlora

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf bank-txn-parser-qlora && git clone -q https://github.com/Tanzil-Ali-Rizvi/bank-txn-parser-qlora
%cd bank-txn-parser-qlora
!pip install -q -U peft bitsandbytes
os.environ["PYTHONPATH"] = "src"
!python -c "import torch, transformers, peft, bitsandbytes; print(torch.__version__, transformers.__version__, peft.__version__, bitsandbytes.__version__)"

## 1. Baselines
A hand-written regex parser (with a dictionary of every merchant seen in training) and the untuned base model with a detailed instruction plus 4 worked examples.

In [ ]:
!python -m txnparser.rules
!python -m txnparser.infer --model Qwen/Qwen2.5-1.5B-Instruct --mode base --name base_fewshot --batch 16
!python -m txnparser.report --systems rules base_fewshot

## 2. QLoRA fine-tune
4-bit NF4 base, LoRA r=16 on all linear layers, 2 epochs over 5,000 examples, loss on the JSON answer only.

In [ ]:
!python -m txnparser.train --data data --out runs/qlora

## 3. Evaluate the fine-tuned model
The fine-tuned model gets a one-line prompt instead of the long instruction and examples.

In [ ]:
!python -m txnparser.infer --model Qwen/Qwen2.5-1.5B-Instruct --adapter runs/qlora/adapter --mode ft --name ft --batch 32 --save-merged runs/merged
!python -m txnparser.report

## 4. Publish to the Hugging Face Hub

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import HfApi
user = HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"]
print("Uploading as", user)
!python -m txnparser.publish_hf --user {user} --merged runs/merged --adapter runs/qlora/adapter

In [ ]:
!mkdir -p /kaggle/working/results && cp -r results/* runs/qlora/train_info.json /kaggle/working/results/ && ls /kaggle/working/results